# Normal Attention with 4 Channels: I, Q, A(t), phase(t)

This notebook trains the new normal-attention model:

```text
mcldnn_attention_iq_amp_phase
```

The model uses one 4-channel sequence input:

```text
[I(t), Q(t), A(t), phase(t)]
```

It compares the new model against the existing normal-attention baseline:

```text
mcldnn_attention
```

Rules followed:

- The new 4-channel attention model is trained from scratch.
- The new model uses standard loss and no class weights.
- The normal-attention baseline is used for comparison only.
- Final zip exports only `experiments/5class_attention_iq_amp_phase/...`.


In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/akshlabh/amr-5-class.git"
WORK_DIR = Path("/kaggle/working/amr-5-class")
DATASET = Path("/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat")


def find_attached_repo():
    """Use an attached repo copy when Kaggle internet/GitHub is unavailable."""
    for root in Path("/kaggle/input").glob("**"):
        if (root / "src" / "train.py").exists() and (root / "configs").exists():
            return root
    return None


if (WORK_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(WORK_DIR), "pull"], check=False)
elif WORK_DIR.exists() and (WORK_DIR / "src" / "train.py").exists():
    print(f"Using existing work dir: {WORK_DIR}")
else:
    attached = find_attached_repo()
    if attached is not None:
        print(f"Copying attached repo from: {attached}")
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print("Cloning repo from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

NORMAL_DIR = Path("experiments/5class_attention")
FOURCH_DIR = Path("experiments/5class_attention_iq_amp_phase")
COMPARE_DIR = FOURCH_DIR / "comparison_with_normal_attention"
MODEL_VARIANT = "normal_attention_4ch_I_Q_A_phase_standard_loss"
VARIANT_FILE = Path("results/model_variant.txt")

required = [
    "src/models/mcldnn_attention.py",
    "src/models/mcldnn_attention_iq_amp_phase.py",
    "src/features/signal_features.py",
    "configs/exp_5class_attention.yaml",
    "configs/exp_5class_attention_iq_amp_phase.yaml",
    "src/train.py",
]

print(f"Working dir: {Path.cwd()}")
print(f"Dataset    : {DATASET}")
print(f"Dataset OK : {DATASET.exists()}")
for f in required:
    print(f"{'OK' if Path(f).exists() else 'MISSING'} {f}")

assert DATASET.exists(), f"Dataset not found: {DATASET}"
for f in required:
    assert Path(f).exists(), f"Required repo file missing: {f}"


In [ ]:
# CELL 2: Verify 4-channel features and model shape
import gc
import keras
import keras.backend as K

keras.mixed_precision.set_global_policy("float32")
K.clear_session(); gc.collect()

from src.features.signal_features import extract_iq_amplitude_phase_channels
from src.models.mcldnn_attention import build_mcldnn_attention
from src.models.mcldnn_attention_iq_amp_phase import (
    build_mcldnn_attention_iq_amp_phase,
    build_mcldnn_attention_iq_amp_phase_extractor,
)

np.random.seed(42)
X_fake = np.random.randn(8, 2, 128).astype("float32")
X4 = extract_iq_amplitude_phase_channels(X_fake)

print(f"4-channel feature shape: {X4.shape}  expected (8, 128, 4)")
print(f"I(t) range            : [{X4[:,:,0].min():.3f}, {X4[:,:,0].max():.3f}]")
print(f"Q(t) range            : [{X4[:,:,1].min():.3f}, {X4[:,:,1].max():.3f}]")
print(f"A(t) range            : [{X4[:,:,2].min():.3f}, {X4[:,:,2].max():.3f}]  expected >= 0")
print(f"phase(t) range        : [{X4[:,:,3].min():.3f}, {X4[:,:,3].max():.3f}]  expected [-1, 1]")

assert X4.shape == (8, 128, 4)
assert X4[:,:,2].min() >= 0
assert X4[:,:,3].min() >= -1.0001 and X4[:,:,3].max() <= 1.0001

normal_model = build_mcldnn_attention(classes=5)
fourch_model = build_mcldnn_attention_iq_amp_phase(classes=5)
fourch_extractor = build_mcldnn_attention_iq_amp_phase_extractor(classes=5)

print(f"Normal attention params  : {normal_model.count_params():,}")
print(f"4-channel attention params: {fourch_model.count_params():,}")
print(f"Under 300k               : {'YES' if fourch_model.count_params() < 300_000 else 'NO'}")

x = np.zeros((4, 128, 4), dtype="float32")
pred = fourch_model.predict([x], verbose=0)
pred2, attn = fourch_extractor.predict([x], verbose=0)

print(f"4-channel training output shape  : {pred.shape}")
print(f"4-channel extractor softmax shape: {pred2.shape}")
print(f"4-channel attention map shape    : {attn.shape}")

assert pred.shape == (4, 5)
assert pred2.shape == (4, 5)
assert attn.shape == (4, 4, 124, 124)
assert fourch_model.count_params() < 300_000
print("Shape checks passed")

del normal_model, fourch_model, fourch_extractor
K.clear_session(); gc.collect()


In [ ]:
# CELL 3: Train 4-channel normal-attention model from scratch
# Existing 4-channel outputs are removed first so stale weights are not reused.

if FOURCH_DIR.exists():
    print(f"Removing existing 4-channel attention outputs: {FOURCH_DIR}")
    shutil.rmtree(FOURCH_DIR)

print("Training 4-channel normal attention model from scratch...\n")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(
    [
        sys.executable, "-u", "src/train.py",
        "--config", "configs/exp_5class_attention_iq_amp_phase.yaml",
        "--datasetpath", str(DATASET),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, process.args)

(FOURCH_DIR / "results").mkdir(parents=True, exist_ok=True)
(FOURCH_DIR / VARIANT_FILE).write_text(MODEL_VARIANT)

print("\n4-channel model training finished.")
print("Checkpoint:", (FOURCH_DIR / "checkpoints" / "best_model.weights.h5").exists())
print("Test score:", (FOURCH_DIR / "results" / "test_score.csv").exists())
print("SNR CSV   :", (FOURCH_DIR / "results" / "acc_per_snr.csv").exists())
print("Variant   :", (FOURCH_DIR / VARIANT_FILE).read_text().strip())


In [ ]:
# CELL 4: Ensure normal-attention baseline results exist
# Baseline is used for comparison only and is not exported in the final zip.

NORMAL_RESULT = NORMAL_DIR / "results" / "test_score.csv"
NORMAL_SNR = NORMAL_DIR / "results" / "acc_per_snr.csv"

if NORMAL_RESULT.exists() and NORMAL_SNR.exists():
    print("Normal attention results already exist. Using them for comparison only.")
else:
    print("Normal attention results not found. Training normal attention for comparison only...")
    subprocess.run([
        sys.executable, "src/train.py",
        "--config", "configs/exp_5class_attention.yaml",
        "--datasetpath", str(DATASET),
    ], check=True)

print("Normal attention test score exists:", NORMAL_RESULT.exists())
print("Normal attention SNR CSV exists  :", NORMAL_SNR.exists())


In [ ]:
# CELL 5: Compare Normal Attention vs 4-channel Attention
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

normal_score = pd.read_csv(NORMAL_DIR / "results" / "test_score.csv")
fourch_score = pd.read_csv(FOURCH_DIR / "results" / "test_score.csv")

normal_snr = pd.read_csv(NORMAL_DIR / "results" / "acc_per_snr.csv").rename(columns={"accuracy": "normal_attention_acc"})
fourch_snr = pd.read_csv(FOURCH_DIR / "results" / "acc_per_snr.csv").rename(columns={"accuracy": "attention_iq_amp_phase_acc"})

comparison = pd.merge(normal_snr, fourch_snr, on="snr", how="inner")
comparison["fourch_minus_normal"] = comparison["attention_iq_amp_phase_acc"] - comparison["normal_attention_acc"]

for col in ["normal_attention_acc", "attention_iq_amp_phase_acc", "fourch_minus_normal"]:
    comparison[col + "_percent"] = 100.0 * comparison[col]

summary = pd.DataFrame([
    {
        "model": "normal_attention",
        "test_loss": float(normal_score.loc[0, "loss"]),
        "test_accuracy": float(normal_score.loc[0, "accuracy"]),
        "mean_snr_accuracy": float(normal_snr["normal_attention_acc"].mean()),
        "peak_snr_accuracy": float(normal_snr["normal_attention_acc"].max()),
        "peak_snr_db": int(normal_snr.loc[normal_snr["normal_attention_acc"].idxmax(), "snr"]),
    },
    {
        "model": "attention_iq_amp_phase",
        "test_loss": float(fourch_score.loc[0, "loss"]),
        "test_accuracy": float(fourch_score.loc[0, "accuracy"]),
        "mean_snr_accuracy": float(fourch_snr["attention_iq_amp_phase_acc"].mean()),
        "peak_snr_accuracy": float(fourch_snr["attention_iq_amp_phase_acc"].max()),
        "peak_snr_db": int(fourch_snr.loc[fourch_snr["attention_iq_amp_phase_acc"].idxmax(), "snr"]),
    },
])

summary_csv = COMPARE_DIR / "normal_vs_attention_iq_amp_phase_summary.csv"
comparison_csv = COMPARE_DIR / "normal_vs_attention_iq_amp_phase_acc_per_snr.csv"
summary.to_csv(summary_csv, index=False)
comparison.to_csv(comparison_csv, index=False)

print("Summary:")
display(summary)
print("\nPer-SNR comparison:")
display(comparison)
print(f"Saved: {summary_csv}")
print(f"Saved: {comparison_csv}")


In [ ]:
# CELL 6: Plot test accuracy vs SNR and delta curve
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(comparison["snr"], comparison["normal_attention_acc_percent"], marker="o", linewidth=2.4, label="Normal attention")
ax.plot(comparison["snr"], comparison["attention_iq_amp_phase_acc_percent"], marker="s", linewidth=2.4, label="4-channel attention: I,Q,A,phase")
ax.set_title("Normal Attention vs 4-channel I/Q/Amplitude/Phase Attention: Accuracy vs SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(comparison["snr"])
plt.tight_layout()
curve_path = COMPARE_DIR / "normal_vs_attention_iq_amp_phase_acc_vs_snr.png"
fig.savefig(curve_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {curve_path}")

fig, ax = plt.subplots(figsize=(11, 4.8))
colors = ["#2ca02c" if x >= 0 else "#d62728" for x in comparison["fourch_minus_normal_percent"]]
ax.bar(comparison["snr"].astype(str), comparison["fourch_minus_normal_percent"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("4-channel Attention minus Normal Attention by SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Delta accuracy (percentage points)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
delta_path = COMPARE_DIR / "attention_iq_amp_phase_minus_normal_delta_by_snr.png"
fig.savefig(delta_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {delta_path}")


In [ ]:
# CELL 7: Compare training accuracy vs SNR curves
# Evaluates saved best weights on the training split, grouped by SNR.

import keras.backend as K
from src.dataset import load_data, FIVE_CLASS
from src.models.mcldnn_attention import build_mcldnn_attention
from src.models.mcldnn_attention_iq_amp_phase import build_mcldnn_attention_iq_amp_phase
from src.features.signal_features import extract_iq_amplitude_phase_channels

K.clear_session(); gc.collect()

(mods_train, snrs_train, lbl_train), (X_train, Y_train), _, _, (train_idx, _, _) = load_data(
    str(DATASET), FIVE_CLASS, seed=2016, shuffle_split=False
)
train_SNRs = np.array([lbl_train[i][1] for i in train_idx])
TRAIN_SNRS = sorted(set(train_SNRs))
print(f"Training set: {X_train.shape[0]} samples across {len(TRAIN_SNRS)} SNRs")


def make_normal_inputs(X):
    return [
        np.expand_dims(X, axis=3).astype("float32"),
        np.expand_dims(X[:, 0, :], axis=2).astype("float32"),
        np.expand_dims(X[:, 1, :], axis=2).astype("float32"),
    ]


def make_fourch_inputs(X):
    return [extract_iq_amplitude_phase_channels(X)]


def slice_inputs(inputs, mask):
    return [arr[mask] for arr in inputs]


def eval_by_snr(model, inputs, Y, snr_labels, snr_values, batch_size=400):
    accs = {}
    for snr in snr_values:
        mask = snr_labels == snr
        score = model.evaluate(slice_inputs(inputs, mask), Y[mask], verbose=0, batch_size=batch_size)
        accs[int(snr)] = float(score[1])
    return accs

normal_inputs = make_normal_inputs(X_train)
fourch_inputs = make_fourch_inputs(X_train)

normal_model = build_mcldnn_attention(classes=5)
normal_model.load_weights(NORMAL_DIR / "checkpoints" / "best_model.weights.h5")
normal_train_acc = eval_by_snr(normal_model, normal_inputs, Y_train, train_SNRs, TRAIN_SNRS)
print("Normal attention train mean acc:", np.mean(list(normal_train_acc.values())))
del normal_model; K.clear_session(); gc.collect()

fourch_model = build_mcldnn_attention_iq_amp_phase(classes=5)
fourch_model.load_weights(FOURCH_DIR / "checkpoints" / "best_model.weights.h5")
fourch_train_acc = eval_by_snr(fourch_model, fourch_inputs, Y_train, train_SNRs, TRAIN_SNRS)
print("4-channel attention train mean acc:", np.mean(list(fourch_train_acc.values())))
del fourch_model; K.clear_session(); gc.collect()

train_comparison = pd.DataFrame({
    "snr": TRAIN_SNRS,
    "normal_attention_train_acc": [normal_train_acc[s] for s in TRAIN_SNRS],
    "attention_iq_amp_phase_train_acc": [fourch_train_acc[s] for s in TRAIN_SNRS],
})
train_comparison["fourch_minus_normal_train"] = train_comparison["attention_iq_amp_phase_train_acc"] - train_comparison["normal_attention_train_acc"]
for col in ["normal_attention_train_acc", "attention_iq_amp_phase_train_acc", "fourch_minus_normal_train"]:
    train_comparison[col + "_percent"] = 100.0 * train_comparison[col]

train_comparison_csv = COMPARE_DIR / "normal_vs_attention_iq_amp_phase_train_acc_per_snr.csv"
train_comparison.to_csv(train_comparison_csv, index=False)
display(train_comparison)
print(f"Saved: {train_comparison_csv}")


In [ ]:
# CELL 8: Plot training accuracy vs SNR and train-vs-test for 4-channel model
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(train_comparison["snr"], train_comparison["normal_attention_train_acc_percent"], marker="o", linewidth=2.4, label="Normal attention train")
ax.plot(train_comparison["snr"], train_comparison["attention_iq_amp_phase_train_acc_percent"], marker="s", linewidth=2.4, label="4-channel attention train")
ax.set_title("Training Accuracy vs SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Training accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(train_comparison["snr"])
plt.tight_layout()
train_curve_path = COMPARE_DIR / "normal_vs_attention_iq_amp_phase_train_acc_vs_snr.png"
fig.savefig(train_curve_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {train_curve_path}")

train_test = pd.merge(
    comparison[["snr", "attention_iq_amp_phase_acc"]],
    train_comparison[["snr", "attention_iq_amp_phase_train_acc"]],
    on="snr",
    how="inner",
)
train_test["fourch_test_acc_percent"] = 100.0 * train_test["attention_iq_amp_phase_acc"]
train_test["fourch_train_acc_percent"] = 100.0 * train_test["attention_iq_amp_phase_train_acc"]
train_test_csv = COMPARE_DIR / "attention_iq_amp_phase_train_test_acc_per_snr.csv"
train_test.to_csv(train_test_csv, index=False)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.plot(train_test["snr"], train_test["fourch_train_acc_percent"], marker="o", linewidth=2.4, label="4-channel train")
ax.plot(train_test["snr"], train_test["fourch_test_acc_percent"], marker="s", linewidth=2.4, label="4-channel test")
ax.set_title("4-channel I/Q/Amplitude/Phase Attention: Train vs Test Accuracy by SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(train_test["snr"])
plt.tight_layout()
train_vs_test_path = COMPARE_DIR / "attention_iq_amp_phase_train_vs_test_acc_by_snr.png"
fig.savefig(train_vs_test_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {train_test_csv}")
print(f"Saved: {train_vs_test_path}")


In [ ]:
# CELL 9: Copy comparison files into 4-channel results folder
# This makes the final zip self-contained without exporting normal baseline checkpoints/results.

FOURCH_COMPARE_EXPORT = FOURCH_DIR / "results" / "comparison_with_normal_attention"
FOURCH_COMPARE_EXPORT.mkdir(parents=True, exist_ok=True)

files_to_copy = [
    summary_csv,
    comparison_csv,
    curve_path,
    delta_path,
    train_comparison_csv,
    train_curve_path,
    train_test_csv,
    train_vs_test_path,
]

for src in files_to_copy:
    dst = FOURCH_COMPARE_EXPORT / Path(src).name
    shutil.copy2(src, dst)
    print(f"Copied: {dst}")


In [ ]:
# CELL 10: Create repo-ready zip for ONLY 4-channel attention results
# The zip contains: experiments/5class_attention_iq_amp_phase/...
# Extract it at repo root and paths line up with other experiment folders.

stamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = Path("/kaggle/working") / f"attention_iq_amp_phase_repo_ready_{stamp}"
zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=str(WORK_DIR),
    base_dir="experiments/5class_attention_iq_amp_phase",
)

print(f"Created repo-ready zip: {zip_path}")
print("\nExtract this zip at the repo root. It will create/update:")
print("  experiments/5class_attention_iq_amp_phase/")
print("\nIncluded files:")
for path in sorted(FOURCH_DIR.rglob("*")):
    if path.is_file():
        print(" -", Path("experiments/5class_attention_iq_amp_phase") / path.relative_to(FOURCH_DIR))

display(FileLink(zip_path))
